# 02 — Reversible variants at the *same* fixed batch size

Same model size, same data, same 50M-token budget, same batch (32 × 256) as the baseline — only the layer-to-layer update rule changes,
and the backward pass reconstructs activations instead of storing them.

| variant | update rule | paper eq. |
|---|---|---|
| `midpoint` | p_{l+1} = p_{l-1} + 2h·f(p_l) | 2.4 |
| `leapfrog` | p_{l+1} = 2p_l − p_{l-1} + h²·f(p_l) | 2.6 |
| `hamiltonian` | q_l = q_{l-1} + Attn(p_{l-1}); p_l = p_{l-1} + MLP(q_l)  (symplectic Euler) | 2.8–2.9 |

Each run takes roughly 1.3–1.6× the baseline time (recompute overhead). Results are saved after **each** variant, so if Colab
disconnects you can restart with a shorter `VARIANTS` list. Expected time on T4: ~40–55 min per variant.

In [ ]:
# @title Setup — clone repo (if needed), install deps, detect GPU
import os, sys, subprocess, json, time, math
REPO_URL = "https://github.com/swatibansal/reversible-llm-poc.git"

if not os.path.exists("src/revllm.py"):
    if os.path.exists("../src/revllm.py"):
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "reversible-llm-poc"], check=True)
        os.chdir("reversible-llm-poc")
sys.path.insert(0, os.path.abspath("src"))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tiktoken", "datasets", "matplotlib"], check=False)

import torch
from revllm import Config, GPT, SavedTensorMeter
from data import prepare_tinystories, prepare_synthetic, TokenStream
import train as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# SMOKE mode = tiny synthetic run that finishes in ~1 min on CPU. Auto-enabled when there is no GPU.
SMOKE = os.environ.get("SMOKE", "0") == "1" or DEVICE == "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu", "| SMOKE mode:", SMOKE)
os.makedirs("results", exist_ok=True)


In [ ]:
# @title Experiment configuration (shared by all notebooks)
if SMOKE:
    MODEL  = dict(vocab_size=512, block_size=64, n_layer=4, n_embd=128, n_head=4)
    TOKENS = 200_000          # token budget
    BATCH  = 16               # the fixed batch size for notebooks 01/02
    LR     = 2e-3
    LOG    = dict(eval_every=100, eval_iters=5, log_every=50)
else:
    # ~20.8M parameters (12.9M in the tied GPT-2 embedding, 7.9M in 10 transformer blocks of width 256)
    MODEL  = dict(vocab_size=50257, block_size=256, n_layer=10, n_embd=256, n_head=4)
    TOKENS = 50_000_000
    BATCH  = 32               # 32 x 256 = 8,192 tokens / step  -> ~6,100 steps for 50M tokens
    LR     = 6e-4
    LOG    = dict(eval_every=250, eval_iters=20, log_every=50)

def get_data():
    if SMOKE:
        return prepare_synthetic("data/synthetic", 2_000_000, 200_000, vocab=MODEL["vocab_size"])
    return prepare_tinystories("data/tinystories", n_train_tokens=55_000_000, n_val_tokens=2_000_000)

def gpu_table(rows, headers):
    w = [max(len(str(r[i])) for r in [headers] + rows) for i in range(len(headers))]
    line = lambda r: "| " + " | ".join(str(c).ljust(w[i]) for i, c in enumerate(r)) + " |"
    print(line(headers)); print("|" + "|".join("-" * (x + 2) for x in w) + "|")
    for r in rows: print(line(r))

import matplotlib.pyplot as plt
def plot_runs(results, key="curve", title="training loss", smooth=25):
    plt.figure(figsize=(8, 4.5))
    for r in results:
        c = r[key]
        if not c: continue
        xs = [p[1] / 1e6 for p in c]; ys = [p[2] for p in c]
        if key == "curve" and smooth > 1 and len(ys) > smooth:
            ys = [sum(ys[max(0, i - smooth):i + 1]) / len(ys[max(0, i - smooth):i + 1]) for i in range(len(ys))]
        plt.plot(xs, ys, label=f"{r['run_name']}  (final {ys[-1]:.3f})")
    plt.xlabel("tokens seen (M)"); plt.ylabel("cross-entropy (nats)"); plt.title(title); plt.legend(); plt.grid(alpha=.3)
    plt.show()


In [ ]:
VARIANTS = ["midpoint", "leapfrog", "hamiltonian"]     # remove entries here to resume a partial run
H = {"midpoint": 0.5, "leapfrog": 1.0, "hamiltonian": 1.0}  # midpoint uses 2h -> 2h=1 matches the baseline's residual scale

train_bin, val_bin = get_data()
results = []
for mode in VARIANTS:
    cfg = Config(mode=mode, h=H[mode], rev_backprop=True, **MODEL)
    res, _ = T.train(cfg, train_bin, val_bin, batch_size=BATCH, tokens_budget=TOKENS, lr=LR,
                     out_json=f"results/{mode}_B{BATCH}.json", **LOG)
    results.append(res)
    if DEVICE == "cuda": torch.cuda.empty_cache()

## Overhead isolation: same architecture, activations *stored* instead of recomputed

A short run (400 steps) of `midpoint` with `rev_backprop=False`. Its loss curve must match the reversible one step-for-step
(same seed, same data order) — proving the memory-free backward changes *only* memory and time, never the math.

In [ ]:
steps_short = 400 if not SMOKE else 40
cfg = Config(mode="midpoint", h=H["midpoint"], rev_backprop=False, **MODEL)
res_norev, _ = T.train(cfg, train_bin, val_bin, batch_size=BATCH, tokens_budget=steps_short * BATCH * MODEL["block_size"], lr=LR,
                       out_json=f"results/midpoint-norev_B{BATCH}_short.json", run_name="midpoint-norev_short", eval_every=10**9, eval_iters=1, log_every=100)
cfg = Config(mode="midpoint", h=H["midpoint"], rev_backprop=True, **MODEL)
res_rev_short, _ = T.train(cfg, train_bin, val_bin, batch_size=BATCH, tokens_budget=steps_short * BATCH * MODEL["block_size"], lr=LR,
                       out_json=f"results/midpoint-rev_B{BATCH}_short.json", run_name="midpoint-rev_short", eval_every=10**9, eval_iters=1, log_every=100)
d = max(abs(a[2] - b[2]) for a, b in zip(res_norev["curve"], res_rev_short["curve"]))
print(f"\nmax |loss difference| over {steps_short} steps: {d:.2e}   (fp16 autocast -> expect ~1e-3 or less)")
print(f"tokens/s  stored activations: {res_norev['tokens_per_s_steady']:,.0f}   recomputed: {res_rev_short['tokens_per_s_steady']:,.0f}"
      f"   -> recompute overhead {(res_norev['tokens_per_s_steady']/res_rev_short['tokens_per_s_steady']-1)*100:.0f}%")
if DEVICE == "cuda":
    print(f"peak GB   stored activations: {res_norev['peak_mem_gb']:.2f}   recomputed: {res_rev_short['peak_mem_gb']:.2f}")

## Compare with the baseline (run notebook 01 first)

In [ ]:
allres = []
for f in sorted(os.listdir("results")):
    if f.endswith(f"_B{BATCH}.json"):
        allres.append(json.load(open("results/" + f)))
rows = [[r["run_name"], f"{r['final_train_loss']:.4f}", f"{r['final_val_loss']:.4f}", f"{r['tokens_per_s_steady']:,.0f}",
         f"{r['peak_mem_gb']:.2f}" if r["peak_mem_gb"] else "n/a", f"{r['wall_time_s']/60:.1f}"] for r in allres]
gpu_table(rows, ["run", "train loss", "val loss", "tokens/s", "peak GB", "minutes"])
plot_runs(allres, title=f"training loss @ batch {BATCH}")
plot_runs(allres, key="val_curve", title=f"validation loss @ batch {BATCH}")
plt.savefig("results/loss_curves_fixed_batch.png", dpi=120)